# TALE ClinVar SNV ISM Logo Notebook (publication-style, v6)

This notebook uses the updated TALE PyTorch model to draw paired Ref / Alt ISM logos around an 89-nt candidate SNV sequence. Edit the final parameter cell to change the SNV, output target, or figure settings.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import logomaker


In [ ]:
# =========================
# Configuration note.
# =========================
MODEL_BUNDLE_PATH = "../model/model_bundle.pt"
GPU_INDEX = 7
USE_EMA = True
SNV_INDEX_0BASED = 44


In [ ]:

# =========================
# Configuration note.
# =========================
VOCAB = {"A": 0, "C": 1, "G": 2, "T": 3, "N": 4, "PAD": 5}
PAD_ID = VOCAB["PAD"]

class AttentionReadout(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.0):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, mask=None, return_attention=False):
        attn_logits = self.score(x).squeeze(-1)
        if mask is not None:
            attn_logits = attn_logits.masked_fill(~mask, -1e9)
        attn_weights = torch.softmax(attn_logits, dim=-1)
        pooled = torch.sum(x * attn_weights.unsqueeze(-1), dim=1)
        if return_attention:
            return pooled, attn_weights
        return pooled


class ConvBlock1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, pool_type="none", pool_size=1, activation="relu"):
        super().__init__()
        self.kernel_size = int(kernel_size)
        self.stride = int(stride)
        self.pool_type = str(pool_type).lower()
        self.pool_size = int(pool_size)
        self.activation = str(activation).lower()

        if self.activation != "relu":
            raise ValueError("Only relu activation is supported.")

        conv_padding = self.kernel_size // 2
        self.conv = nn.Conv1d(
            in_channels=int(in_channels),
            out_channels=int(out_channels),
            kernel_size=self.kernel_size,
            stride=self.stride,
            padding=conv_padding
        )
        self.act = nn.ReLU()

        if self.pool_type == "none":
            self.pool = nn.Identity()
        elif self.pool_type == "max":
            self.pool = nn.MaxPool1d(kernel_size=self.pool_size, stride=self.pool_size)
        else:
            raise ValueError("pool_type must be 'none' or 'max'.")

    def _conv_out_len(self, seq_lens):
        seq_lens = torch.as_tensor(seq_lens)
        padding = self.kernel_size // 2
        return torch.div(seq_lens + 2 * padding - self.kernel_size, self.stride, rounding_mode="floor") + 1

    def _pool_out_len(self, seq_lens):
        if self.pool_type == "none":
            return seq_lens
        return torch.div(seq_lens - self.pool_size, self.pool_size, rounding_mode="floor") + 1

    def output_lengths(self, seq_lens):
        seq_lens = self._conv_out_len(seq_lens)
        seq_lens = torch.clamp(seq_lens, min=1)
        seq_lens = self._pool_out_len(seq_lens)
        seq_lens = torch.clamp(seq_lens, min=1)
        return seq_lens

    def output_length_scalar(self, seq_len):
        seq_len = int(seq_len)
        padding = self.kernel_size // 2
        seq_len = (seq_len + 2 * padding - self.kernel_size) // self.stride + 1
        seq_len = max(seq_len, 1)
        if self.pool_type == "max":
            seq_len = (seq_len - self.pool_size) // self.pool_size + 1
            seq_len = max(seq_len, 1)
        return seq_len

    def forward(self, x, seq_lens=None):
        x = self.conv(x)
        x = self.act(x)
        x = self.pool(x)
        if seq_lens is not None:
            seq_lens = self.output_lengths(seq_lens).to(x.device)
            mask = torch.arange(x.size(-1), device=x.device).unsqueeze(0) < seq_lens.unsqueeze(1)
            x = x * mask.unsqueeze(1).to(x.dtype)
        return x, seq_lens


class TALELSTMRegressor(nn.Module):
    def __init__(
        self,
        vocab_size=6,
        pad_id=5,
        embed_dim=5,
        max_len=46,
        use_positional_embedding=True,
        positional_dropout=0.0,
        use_cnn_before_lstm=False,
        cnn_out_channels=None,
        cnn_kernel_sizes=None,
        cnn_strides=None,
        cnn_pool_types=None,
        cnn_pool_sizes=None,
        cnn_activation="relu",
        cnn_dropout=0.0,
        lstm_hidden_dims=None,
        lstm_bidirectional=None,
        lstm_inter_layer_dropout=0.0,
        lstm_output_dropout=0.0,
        use_packed_lstm=True,
        readout_mode="flatten",
        attention_hidden_dim=64,
        fc_hidden_dims=None,
        fc_inter_layer_dropout=0.0,
        fc_output_dropout=0.0,
        output_dim=4
    ):
        super().__init__()
        self.max_len = int(max_len)
        self.pad_id = int(pad_id)
        self.use_positional_embedding = bool(use_positional_embedding)
        self.use_cnn_before_lstm = bool(use_cnn_before_lstm)
        self.use_packed_lstm = bool(use_packed_lstm)
        self.readout_mode = str(readout_mode).lower()

        lstm_hidden_dims = list(lstm_hidden_dims or [])
        lstm_bidirectional = list(lstm_bidirectional or [])
        fc_hidden_dims = list(fc_hidden_dims or [])
        cnn_out_channels = list(cnn_out_channels or [])
        cnn_kernel_sizes = list(cnn_kernel_sizes or [])
        cnn_strides = list(cnn_strides or [])
        cnn_pool_types = list(cnn_pool_types or [])
        cnn_pool_sizes = list(cnn_pool_sizes or [])

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=self.pad_id)

        if self.use_positional_embedding:
            self.positional_embedding = nn.Embedding(max_len, embed_dim)
            self.positional_dropout = nn.Dropout(positional_dropout)
        else:
            self.positional_embedding = None
            self.positional_dropout = nn.Identity()

        self.cnn_blocks = nn.ModuleList()
        self.cnn_dropout = nn.Dropout(float(cnn_dropout))
        current_input_size = embed_dim
        self.sequence_output_len = self.max_len

        if self.use_cnn_before_lstm:
            for out_channels, kernel_size, stride, pool_type, pool_size in zip(
                cnn_out_channels, cnn_kernel_sizes, cnn_strides, cnn_pool_types, cnn_pool_sizes
            ):
                block = ConvBlock1d(
                    in_channels=current_input_size,
                    out_channels=int(out_channels),
                    kernel_size=int(kernel_size),
                    stride=int(stride),
                    pool_type=str(pool_type),
                    pool_size=int(pool_size),
                    activation=str(cnn_activation)
                )
                self.cnn_blocks.append(block)
                current_input_size = int(out_channels)
                self.sequence_output_len = block.output_length_scalar(self.sequence_output_len)

        self.lstm_layers = nn.ModuleList()
        self.lstm_inter_dropouts = nn.ModuleList()
        self.lstm_output_dim = None
        for layer_idx, (hidden_size, is_bidirectional) in enumerate(zip(lstm_hidden_dims, lstm_bidirectional)):
            lstm_layer = nn.LSTM(
                input_size=current_input_size,
                hidden_size=int(hidden_size),
                batch_first=True,
                bidirectional=bool(is_bidirectional)
            )
            self.lstm_layers.append(lstm_layer)
            current_input_size = int(hidden_size) * (2 if bool(is_bidirectional) else 1)
            self.lstm_output_dim = current_input_size
            self.lstm_inter_dropouts.append(
                nn.Dropout(float(lstm_inter_layer_dropout)) if layer_idx < len(lstm_hidden_dims) - 1 else nn.Identity()
            )

        self.lstm_output_dropout = nn.Dropout(float(lstm_output_dropout))

        if self.readout_mode == "flatten":
            current_fc_in = self.sequence_output_len * self.lstm_output_dim
            self.readout = None
        else:
            self.readout = AttentionReadout(
                input_dim=self.lstm_output_dim,
                hidden_dim=attention_hidden_dim,
                dropout=float(fc_inter_layer_dropout)
            )
            current_fc_in = self.lstm_output_dim

        fc_layers = []
        for layer_idx, hidden_dim in enumerate(fc_hidden_dims):
            fc_layers.append(nn.Linear(current_fc_in, int(hidden_dim)))
            fc_layers.append(nn.ReLU())
            if layer_idx < len(fc_hidden_dims) - 1 and float(fc_inter_layer_dropout) > 0:
                fc_layers.append(nn.Dropout(float(fc_inter_layer_dropout)))
            current_fc_in = int(hidden_dim)

        self.fc_stack = nn.Sequential(*fc_layers) if len(fc_layers) > 0 else nn.Identity()
        self.fc_output_dropout = nn.Dropout(float(fc_output_dropout))
        self.fc_out = nn.Linear(current_fc_in, output_dim)

    def _run_cnn_frontend(self, x, seq_lens):
        if not self.use_cnn_before_lstm:
            return x, seq_lens
        mask = torch.arange(x.size(1), device=x.device).unsqueeze(0) < seq_lens.unsqueeze(1)
        x = x * mask.unsqueeze(-1).to(x.dtype)
        x = x.transpose(1, 2)
        for block in self.cnn_blocks:
            x, seq_lens = block(x, seq_lens=seq_lens)
            x = self.cnn_dropout(x)
        x = x.transpose(1, 2)
        return x, seq_lens

    def _run_single_lstm(self, lstm_layer, x, seq_lens):
        if self.use_packed_lstm:
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths=seq_lens.detach().cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = lstm_layer(packed)
            x, _ = nn.utils.rnn.pad_packed_sequence(
                packed_out,
                batch_first=True,
                total_length=self.sequence_output_len if self.use_cnn_before_lstm else self.max_len
            )
        else:
            x, _ = lstm_layer(x)
        return x

    def _run_stacked_lstm(self, x, seq_lens):
        for layer_idx, lstm_layer in enumerate(self.lstm_layers):
            x = self._run_single_lstm(lstm_layer, x, seq_lens)
            x = self.lstm_inter_dropouts[layer_idx](x)
        return x

    def forward(self, x, seq_lens=None, return_attention=False):
        if seq_lens is None:
            seq_lens = (x != self.pad_id).sum(dim=1)
        seq_lens = seq_lens.to(x.device)
        x = self.embedding(x)
        if self.positional_embedding is not None:
            pos_ids = torch.arange(self.max_len, device=x.device).unsqueeze(0).expand(x.size(0), -1)
            x = x + self.positional_embedding(pos_ids)
            x = self.positional_dropout(x)
        x, seq_lens = self._run_cnn_frontend(x, seq_lens)
        x = self._run_stacked_lstm(x, seq_lens)
        x = self.lstm_output_dropout(x)
        if self.readout_mode == "flatten":
            x = x.contiguous().view(x.size(0), -1)
            attn_weights = None
        else:
            mask = torch.arange(x.size(1), device=x.device).unsqueeze(0) < seq_lens.unsqueeze(1)
            x, attn_weights = self.readout(x, mask=mask, return_attention=True)
        x = self.fc_stack(x)
        x = self.fc_output_dropout(x)
        x = self.fc_out(x)
        if return_attention:
            return x, attn_weights
        return x


def build_model_from_config(cfg):
    return TALELSTMRegressor(
        vocab_size=cfg["vocab_size"],
        pad_id=cfg.get("pad_id", PAD_ID),
        embed_dim=cfg["embed_dim"],
        max_len=cfg["max_len"],
        use_positional_embedding=cfg.get("use_positional_embedding", False),
        positional_dropout=cfg.get("positional_dropout", 0.0),
        use_cnn_before_lstm=cfg.get("use_cnn_before_lstm", False),
        cnn_out_channels=cfg.get("cnn_out_channels", []),
        cnn_kernel_sizes=cfg.get("cnn_kernel_sizes", []),
        cnn_strides=cfg.get("cnn_strides", []),
        cnn_pool_types=cfg.get("cnn_pool_types", []),
        cnn_pool_sizes=cfg.get("cnn_pool_sizes", []),
        cnn_activation=cfg.get("cnn_activation", "relu"),
        cnn_dropout=cfg.get("cnn_dropout", 0.0),
        lstm_hidden_dims=cfg["lstm_hidden_dims"],
        lstm_bidirectional=cfg["lstm_bidirectional"],
        lstm_inter_layer_dropout=cfg.get("lstm_inter_layer_dropout", 0.0),
        lstm_output_dropout=cfg.get("lstm_output_dropout", 0.0),
        use_packed_lstm=cfg.get("use_packed_lstm", True),
        readout_mode=cfg.get("readout_mode", "flatten"),
        attention_hidden_dim=cfg.get("attention_hidden_dim", 64),
        fc_hidden_dims=cfg.get("fc_hidden_dims", []),
        fc_inter_layer_dropout=cfg.get("fc_inter_layer_dropout", 0.0),
        fc_output_dropout=cfg.get("fc_output_dropout", 0.0),
        output_dim=cfg["output_dim"],
    )


In [ ]:
# =========================
# 3) Load model bundle
# =========================
DEVICE = torch.device(f"cuda:{GPU_INDEX}" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(MODEL_BUNDLE_PATH, map_location=DEVICE, weights_only=False)

model_config = checkpoint["model_config"]
preprocess_config = checkpoint["preprocess_config"]
selected_weight_set = checkpoint.get("selected_weight_set", "raw")

target_cols = list(preprocess_config["target_cols"])
target_mean = np.array(preprocess_config["target_mean"], dtype=np.float32)
target_std  = np.array(preprocess_config["target_std"],  dtype=np.float32)
max_len = int(preprocess_config["max_len"])
target_to_idx = {name: i for i, name in enumerate(target_cols)}

model = build_model_from_config(model_config).to(DEVICE)
if USE_EMA and checkpoint.get("ema_model_state_dict") is not None:
    model.load_state_dict(checkpoint["ema_model_state_dict"])
    loaded_weight_set = "ema"
else:
    model.load_state_dict(checkpoint["model_state_dict"])
    loaded_weight_set = "raw"
model.eval()

print("Loaded weight set:", loaded_weight_set)
print("Selected weight set in bundle:", selected_weight_set)
print("Target columns:", target_cols)


In [ ]:
# =========================
# Configuration note.
# =========================
BASES = ["A", "C", "G", "T"]


def encode_sequence(seq, max_len=max_len):
    seq = str(seq).upper().strip()
    token_ids = [VOCAB.get(ch, VOCAB["N"]) for ch in seq[:max_len]]
    seq_len = min(len(token_ids), max_len)
    if len(token_ids) < max_len:
        token_ids.extend([PAD_ID] * (max_len - len(token_ids)))
    return token_ids, seq_len


def batch_predict(seqs):
    encoded = [encode_sequence(s, max_len=max_len) for s in seqs]
    x = torch.tensor([e[0] for e in encoded], dtype=torch.long, device=DEVICE)
    seq_lens = torch.tensor([e[1] for e in encoded], dtype=torch.long, device=DEVICE)
    with torch.inference_mode():
        y_scaled = model(x, seq_lens=seq_lens).detach().cpu().numpy()
    y = y_scaled * target_std + target_mean
    return pd.DataFrame(y, columns=target_cols)


def kmerize(seq, k=45):
    return [seq[i:i + k] for i in range(len(seq) - k + 1)]


def predict_big(seq, k=45):
    kmers = kmerize(seq, k=k)
    return batch_predict(kmers)


def mutate_base(seq, pos, alt_base):
    seq = list(seq)
    seq[pos] = alt_base
    return ''.join(seq)


def validate_ref_and_make_alt(ref89, alt_base, snv_idx=SNV_INDEX_0BASED):
    ref89 = str(ref89).upper().strip()
    alt_base = str(alt_base).upper().strip()
    assert len(ref89) == 89, "Invalid value."
    assert alt_base in BASES, "Invalid value."
    ref_center = ref89[snv_idx]
    assert ref_center in BASES, f"ref center base is not A/C/G/T; value is {ref_center}"
    assert alt_base != ref_center, f"alt_base is identical to the ref center base: {alt_base}"
    alt89 = mutate_base(ref89, snv_idx, alt_base)
    return ref89, alt89


def check_target_name(target_name):
    assert target_name in target_to_idx, "Invalid value."


def ism_logo_df(seq89, target_name, k=45):
    """
    Compute an ISM logo matrix for an 89-nt sequence.
    """
    check_target_name(target_name)
    seq89 = str(seq89).upper().strip()
    assert len(seq89) == 89

    wt_pred = predict_big(seq89, k=k)[target_name].values
    rows = []

    for pos, wt in enumerate(seq89):
        if wt not in BASES:
            rows.append({b: 0.0 for b in BASES})
            continue

        mut_effects = []
        for alt in BASES:
            if alt == wt:
                continue
            mut89 = mutate_base(seq89, pos, alt)
            mut_pred = predict_big(mut89, k=k)[target_name].values
            delta = wt_pred - mut_pred
            mut_effects.append(np.median(delta))

        median_effect = float(np.median(mut_effects))
        row = {b: 0.0 for b in BASES}
        row[wt] = median_effect
        rows.append(row)

    df = pd.DataFrame(rows, columns=BASES)
    df.index = np.arange(-44, 45)
    return df


In [ ]:
# =========================
# Configuration note.
# =========================

def style_logo_axis(
    ax,
    ymax,
    *,
    title="",
    is_bottom=False,
    show_x_position_labels=True,
    show_x_axis_label=True,
    x_ticks=(-44, -20, 0, 20, 44),
    x_label="Position relative to SNV",
    y_label="ISM effect",
    title_fontsize=13,
    label_fontsize=13,
    tick_fontsize=13,
    center_line_width=1.2,
    center_line_alpha=0.8,
):
    # Configuration note.
    # Configuration note.
    ax.axvline(
        0,
        linestyle='--',
        linewidth=center_line_width,
        alpha=center_line_alpha,
        color='black',
        zorder=0,
    )
    ax.set_xlim(-44.5, 44.5)
    ax.set_ylim(-ymax, ymax)

    ax.set_title(title, fontsize=title_fontsize, pad=8)
    ax.set_ylabel(y_label, fontsize=label_fontsize)

    # Configuration note.
    ax.set_yticks([-ymax, ymax])
    ax.set_yticklabels([f"{-ymax:.2f}", f"{ymax:.2f}"], fontsize=tick_fontsize)

    # Configuration note.
    if is_bottom and show_x_position_labels:
        ax.set_xticks(list(x_ticks))
        ax.set_xticklabels([str(x) for x in x_ticks], fontsize=tick_fontsize)
        if show_x_axis_label:
            ax.set_xlabel(x_label, fontsize=label_fontsize)
    else:
        ax.set_xticks([])

    # Configuration note.
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(True)
    ax.spines['left'].set_linewidth(1.0)

    ax.tick_params(axis='y', length=3, width=1)
    if is_bottom and show_x_position_labels:
        ax.tick_params(axis='x', bottom=True, labelbottom=True, length=4, width=1, pad=4)
    else:
        ax.tick_params(axis='x', bottom=False, labelbottom=False)


def plot_ref_alt_ism_logo(
    ref89,
    alt_base,
    *,
    target_name="cyt.score.A549",
    figsize=(6, 3),
    title_fontsize=13,
    label_fontsize=13,
    tick_fontsize=13,
    logo_font_name="DejaVu Sans",
    show_x_position_labels=True,
    show_x_axis_label=True,
    x_ticks=(-44, -20, 0, 20, 44),
    x_label="Position relative to SNV",
    y_label="ISM effect",
    y_padding=1.10,
    center_line_width=1.2,
    center_line_alpha=0.8,
    layout_left=0.09,
    layout_right=0.995,
    layout_top=0.92,
    layout_bottom=0.14,
    layout_hspace=0.34,
):
    check_target_name(target_name)
    ref89, alt89 = validate_ref_and_make_alt(ref89, alt_base)
    ref_logo = ism_logo_df(ref89, target_name=target_name)
    alt_logo = ism_logo_df(alt89, target_name=target_name)

    ymax = max(np.abs(ref_logo.values).max(), np.abs(alt_logo.values).max())
    ymax = max(float(ymax), 1e-6) * float(y_padding)

    fig, axes = plt.subplots(
        2,
        1,
        figsize=figsize,
        sharex=True,
        sharey=True,
        gridspec_kw={'hspace': 0.22},
    )

    # Configuration note.
    logomaker.Logo(ref_logo, ax=axes[0], font_name=logo_font_name)
    style_logo_axis(
        axes[0],
        ymax,
        title=f"Ref (center = {ref89[SNV_INDEX_0BASED]})",
        is_bottom=False,
        show_x_position_labels=show_x_position_labels,
        show_x_axis_label=show_x_axis_label,
        x_ticks=x_ticks,
        x_label=x_label,
        y_label=y_label,
        title_fontsize=title_fontsize,
        label_fontsize=label_fontsize,
        tick_fontsize=tick_fontsize,
        center_line_width=center_line_width,
        center_line_alpha=center_line_alpha,
    )

    logomaker.Logo(alt_logo, ax=axes[1], font_name=logo_font_name)
    style_logo_axis(
        axes[1],
        ymax,
        title=f"Alt (center = {alt89[SNV_INDEX_0BASED]})",
        is_bottom=True,
        show_x_position_labels=show_x_position_labels,
        show_x_axis_label=show_x_axis_label,
        x_ticks=x_ticks,
        x_label=x_label,
        y_label=y_label,
        title_fontsize=title_fontsize,
        label_fontsize=label_fontsize,
        tick_fontsize=tick_fontsize,
        center_line_width=center_line_width,
        center_line_alpha=center_line_alpha,
    )

    # Configuration note.
    fig.subplots_adjust(
        left=layout_left,
        right=layout_right,
        top=layout_top,
        bottom=layout_bottom,
        hspace=layout_hspace,
    )

    plt.show()

    print(
        f"Check: logo x-axis is -44..44; center SNV is x=0; "
        f"target={target_name}; Ref center={ref89[SNV_INDEX_0BASED]}, "
        f"Alt center={alt89[SNV_INDEX_0BASED]}, shared ymax={ymax:.3f}"
    )
    return ref_logo, alt_logo


In [ ]:
# =========================
# Configuration note.
# =========================

# ---- SNV input ----
REF89 = "CCAAGGACGTGGCGTCCCTCAGTTCCCAGCTCCAGGACACCCAGGTGAGTGTCCTGCCACATCATCCAGGGGACCTGGGGGGTGGCCTT"
ALT_BASE = "C"
TARGET_NAME = "cyt.score.A549"

# ---- Figure size and fonts ----
FIGSIZE = (6, 3)
TITLE_FONTSIZE = 13
LABEL_FONTSIZE = 13
TICK_FONTSIZE = 13
LOGO_FONT_NAME = "DejaVu Sans"

# ---- Axes and labels ----
SHOW_X_POSITION_LABELS = True
SHOW_X_AXIS_LABEL = True
X_TICKS = [-44, -20, 0, 20, 44]
X_LABEL = "Position relative to SNV"
Y_LABEL = "ISM effect"

# Configuration note.
Y_PADDING = 1.10
CENTER_LINE_WIDTH = 1.2
CENTER_LINE_ALPHA = 0.8
LAYOUT_LEFT = 0.09
LAYOUT_RIGHT = 0.995
LAYOUT_TOP = 0.92
LAYOUT_BOTTOM = 0.14
LAYOUT_HSPACE = 0.34

ref_logo_df, alt_logo_df = plot_ref_alt_ism_logo(
    ref89=REF89,
    alt_base=ALT_BASE,
    target_name=TARGET_NAME,
    figsize=FIGSIZE,
    title_fontsize=TITLE_FONTSIZE,
    label_fontsize=LABEL_FONTSIZE,
    tick_fontsize=TICK_FONTSIZE,
    logo_font_name=LOGO_FONT_NAME,
    show_x_position_labels=SHOW_X_POSITION_LABELS,
    show_x_axis_label=SHOW_X_AXIS_LABEL,
    x_ticks=X_TICKS,
    x_label=X_LABEL,
    y_label=Y_LABEL,
    y_padding=Y_PADDING,
    center_line_width=CENTER_LINE_WIDTH,
    center_line_alpha=CENTER_LINE_ALPHA,
    layout_left=LAYOUT_LEFT,
    layout_right=LAYOUT_RIGHT,
    layout_top=LAYOUT_TOP,
    layout_bottom=LAYOUT_BOTTOM,
    layout_hspace=LAYOUT_HSPACE,
)
